This notebook is for generating "silver label" examples using the trained span identification and technique classification models in order to train a lighter weight model. The raw news article data pre-adding silver labels is from the English-only subset of the Common Crawl News dataset.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path
from datasets import load_dataset
import os
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification

In [2]:
#Identify base directory to ensure portability
BASE_DIR = Path.cwd().resolve().parent
MODELS_DIR = BASE_DIR / "models"
DATA_PATH = BASE_DIR / "data" / "processed" / "semeval_tc_cleaned.csv"

SI_DIR = MODELS_DIR / "semeval_roberta_scanner"
SI_SPEC_DIR = MODELS_DIR / "semeval_roberta_scanner_specialist"
TC_DIR = MODELS_DIR / "semeval_roberta_classifier"

SI_MODEL_PATH = f"{os.fspath(SI_DIR.absolute())}"
SI_SPEC_PATH = f"{os.fspath(SI_SPEC_DIR.absolute())}"
TC_MODEL_PATH = f"{os.fspath(TC_DIR.absolute())}"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [3]:
#Run training notebooks if models are missing
REQUIRED_FILES = ["config.json", "model.safetensors"]

def model_exists(path):
    path = Path(path)
    has_weights = any(path.glob("*.bin")) or any(path.glob("*.safetensors"))
    return has_weights

if not model_exists(SI_MODEL_PATH):
    print("SI Model missing. Running training notebook...")
    %run 4.1-fp-semeval-si-modeling.ipynb
if not model_exists(TC_MODEL_PATH):
    print("TC Model missing. Running training notebook...")
    %run 4.2-fp-semeval-tc-modeling.ipynb

In [4]:
#Load Base SI Model (RoBERTa token-classifier for span detection)
print(f"Loading Base SI Model from: {SI_MODEL_PATH}...")
si_tokenizer = AutoTokenizer.from_pretrained(SI_MODEL_PATH)
si_model = AutoModelForTokenClassification.from_pretrained(SI_MODEL_PATH, local_files_only=True).to(device)
si_model.eval()

#Load Specialist SI Model
print(f"Loading Specialist SI Model from: {SI_SPEC_PATH}...")
si_spec_model = AutoModelForTokenClassification.from_pretrained(SI_SPEC_PATH, local_files_only=True).to(device)
si_spec_model.eval()

Loading Base SI Model from: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_scanner...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading Specialist SI Model from: /Users/frankiepike/MADS_Projects/Claim-Aware_Propaganda_Scanner/models/semeval_roberta_scanner_specialist...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForTokenClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (L

In [5]:
#Load TC Model (Technique Classification)
tc_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
tc_model = AutoModelForSequenceClassification.from_pretrained(TC_MODEL_PATH).to(device)
tc_model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50267, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.2, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [6]:
#Set thresholds for each technique
OPTIMIZED_THRESHOLDS = {
    'Appeal_to_Authority': 0.60,
    'Appeal_to_fear-prejudice': 0.50,
    'Bandwagon_Reductio_ad_hitlerum': 0.10,
    'Black-and-White_Fallacy': 0.20,
    'Causal_Oversimplification': 0.20,
    'Doubt': 0.35,
    'Exaggeration_Minimisation': 0.40,
    'Flag-Waving': 0.45,
    'Loaded_Language': 0.40,
    'Name_Calling_Labeling': 0.55,
    'Repetition': 0.40,
    'Slogans': 0.30,
    'Thought-terminating_Cliches': 0.15,
    'Whataboutism_Straw_Men_Red_Herring': 0.15
}

In [7]:
def run_pipeline(text):
    #Identify spans
    inputs = si_tokenizer(text, return_tensors="pt", truncation=True, padding=True, return_offsets_mapping=True).to(device)
    offsets = inputs.pop("offset_mapping")[0]

    with torch.no_grad():
        #Get Base Model Predictions (3 classes: 0, 1, 2)
        base_outputs = si_model(**inputs)
        base_logits = base_outputs.logits
        base_preds = torch.argmax(base_logits, dim=-1)[0]

        #Get Specialist Model Predictions (2 classes: 0, 1)
        spec_outputs = si_spec_model(**inputs)
        spec_logits = spec_outputs.logits

        #Calculate Specialist Confidence (Softmax)
        spec_probs = F.softmax(spec_logits, dim=-1)[0]
        propaganda_prob = spec_probs[:, 1]

    #Initialize final predictions with base model's results
    final_preds = base_preds.clone()

    #CASCADE LOGIC: If base found nothing (0) AND specialist is confident (> threshold)
    #We set it to '1' (which is the 'Beginning' tag in the base model)
    mask = (base_preds == 0) & (propaganda_prob > 0.5)
    final_preds[mask] = 1

    #Extract spans from the combined predictions
    predicted_spans = []
    current_span = None

    for i, pred in enumerate(final_preds):
        label = pred.item()
        start, end = offsets[i]
        if start == end: continue

        #In BIO tagging: 1 is 'B' (Begin), 2 is 'I' (Inside)
        if label in [1, 2]:
            if current_span is None:
                current_span = [start.item(), end.item()]
            else:
                current_span[1] = end.item()
        else:
            if current_span:
                predicted_spans.append(tuple(current_span))
                current_span = None

    if current_span:
        predicted_spans.append(tuple(current_span))

    #Technique classification
    final_results = []
    for span in predicted_spans:
        span_text = text[span[0]:span[1]].strip()
        if not span_text: continue

        tc_inputs = tc_tokenizer(span_text, return_tensors="pt", truncation=True, padding=True).to(device)

        with torch.no_grad():
            tc_logits = tc_model(**tc_inputs).logits
            probs = torch.sigmoid(tc_logits)[0] #Convert logits to probabilities

        #Find all techniques that pass their specific optimized threshold
        found_techniques = []
        for class_id, prob in enumerate(probs):
            tech_name = tc_model.config.id2label[class_id]
            threshold = OPTIMIZED_THRESHOLDS.get(tech_name, 0.5) # Default to 0.5 just in case

            if prob.item() >= threshold:
                found_techniques.append(tech_name)

        #Fallback: if the model is so unsure that NOTHING passes, just pick the single most likely one
        if not found_techniques:
            best_class = torch.argmax(probs).item()
            found_techniques.append(tc_model.config.id2label[best_class])

        #Add all passing techniques to the final results
        for tech in found_techniques:
            final_results.append({"span": tuple(span), "technique": tech})

    return final_results

In [8]:
#Load the news article dataset
news = load_dataset("vblagoje/cc_news", split="train")
news = news.to_pandas()
news.head()

,title,text,domain,date,description,url,image_url
0,Daughter Duo is Dancing in The Same Company,There's a surprising twist to Regina Willoughb...,www.pointemagazine.com,2017-12-11 20:19:05,There's a surprising twist to Regina Willoughb...,http://www.pointemagazine.com/mother-daughter-...,https://pointe-img.rbl.ms/simage/https%3A%2F%2...
1,New York City Ballet Announces Interim Leaders...,The New York City Ballet Board of Directors an...,www.pointemagazine.com,2017-12-11 17:02:55,NYCB has announced an interim leadership team ...,http://www.pointemagazine.com/nycb-interim-lea...,https://pointe-img.rbl.ms/simage/https%3A%2F%2...
2,Watch Pennsylvania Ballet & Boston Ballet Face...,The Philadelphia Eagles and the New England Pa...,www.pointemagazine.com,2018-02-02 21:58:13,The Philadelphia Eagles and the New England Pa...,http://www.pointemagazine.com/watch-pennsylvan...,https://pointe-img.rbl.ms/simage/https%3A%2F%2...
3,dance shoes,Looking for your next audition shoe? Shot at a...,www.pointemagazine.com,2018-04-24 19:00:11,Looking for your next audition shoe? Shot at a...,https://www.pointemagazine.com/dance-shoes-256...,https://pointe-img.rbl.ms/simage/https%3A%2F%2...
4,Rebecca Krohn on Her Retirement from New York ...,New York City Ballet principal dancer Rebecca ...,www.pointemagazine.com,2017-10-06 14:44:51,We interviewed New York City Ballet principal ...,http://www.pointemagazine.com/rebecca-krohn-re...,https://pointe-img.rbl.ms/simage/https%3A%2F%2...


In [9]:
#Constrain to only the text, as that's the only input our extension will be given
news = news[['text']]

In [ ]:
#Use run pipeline function to get predicted propaganda spans and labels from all the text
news["propaganda"] = news.apply(lambda row: run_pipeline(row["text"]), axis=1)
news.head()